# Modeling (Raw Data)

## Imports

In [1]:
import copy
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

from utils.experiment_separator import (
    get_experiments,
    print_experiment_report,
    separate_experiments,
)

## Configuration

### Data

In [2]:
data_path = "../data/master_ml_raw.xlsx"
column_names = ["density", "cutting_speed", "feed_rate", "depth", "axial_force", "cutting_force"]
raw_column_map = {
    "Density (PCF)": "density",
    "Cutting Speed (RPM)": "cutting_speed",
    "Feed Rate (mm/min)": "feed_rate",
    "Depth (mm)": "depth",
    "Axial Force (N)": "axial_force",
    "Cutting Force (Nmm)": "cutting_force",
}
target = "cutting_force"
input_columns = ["density", "cutting_speed", "feed_rate", "depth", "axial_force"]

### Split and Training

In [3]:
split_seed = 42
n_splits = 5
test_fold = 0
training_seed = 42
run_single_model = False
epochs = 30
patience = 5
min_delta = 0.0
scheduler_factor = 0.5
scheduler_patience = 2

### PyTorch

In [4]:
torch_num_threads = 8
torch_num_interop_threads = 2
input_size = len(input_columns)
output_size = 1

### Model Configurations

#### LSTM

In [5]:
lstm_config = {
    "window_size": 100, "stride": 5, "hidden_size": 128,
    "num_layers": 1, "batch_size": 64, "lr": 1e-3,
    "dropout": 0.0, "optimizer": "Adam", "weight_decay": 0.0,
    "gradient_clip": None,
}

#### GRU

In [6]:
gru_config = {
    "window_size": 75, "stride": 10, "hidden_size": 128,
    "num_layers": 2, "batch_size": 128, "lr": 2e-3,
    "dropout": 0.3, "optimizer": "Adam", "weight_decay": 0.0,
    "gradient_clip": None,
}

#### RNN

In [7]:
rnn_config = {
    "window_size": 100, "stride": 5, "hidden_size": 128,
    "num_layers": 3, "batch_size": 128, "lr": 1e-3,
    "dropout": 0.0, "optimizer": "Adam", "weight_decay": 0.0,
    "gradient_clip": None,
}

#### TCN

In [8]:
tcn_config = {
    "window_size": 76, "stride": 10, "channels": [64] * 5,
    "kernel_size": 3, "batch_size": 128, "lr": 1e-3,
    "dropout": 0.1, "optimizer": "AdamW", "weight_decay": 1e-4,
    "gradient_clip": 1.0,
}
model_configs = {
    "LSTM": lstm_config, "GRU": gru_config,
    "RNN": rnn_config, "TCN": tcn_config,
}

## Definitions

### Configure Runtime

In [9]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def configure_runtime():
    torch.set_num_threads(torch_num_threads)
    try:
        torch.set_num_interop_threads(torch_num_interop_threads)
    except RuntimeError:
        pass
    torch.backends.mkldnn.enabled = True

### Apply Scaling

In [10]:
def apply_scaling(df: pd.DataFrame, scaler: StandardScaler, fit: bool = False) -> pd.DataFrame:
    cols = input_columns + [target]
    if fit:
        df[cols] = scaler.fit_transform(df[cols])
    else:
        df[cols] = scaler.transform(df[cols])
    return df

### Pipeline

In [11]:
class Pipeline:
    def __init__(self):
        self.scaler = StandardScaler()
 
    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        temp = df.copy()
        temp = apply_scaling(temp, self.scaler, fit=True)
        return temp
 
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        temp = df.copy()
        temp = apply_scaling(temp, self.scaler, fit=False)
        return temp

### Split Experiment Keys

In [12]:
def _assign_folds_no_combo_clustering(keys, n_splits, seed):
    rng = np.random.default_rng(seed)
    combos = {}
    for key in keys:
        combos.setdefault(key[:3], []).append(key)
    combo_items = list(combos.items())
    rng.shuffle(combo_items)

    fold_count = {}
    fold_of = {}
    for combo, reps in combo_items:
        density = combo[0]
        counts = fold_count.setdefault(density, [0] * n_splits)
        tie_break = rng.random(n_splits)
        fold_order = sorted(range(n_splits), key=lambda f: (counts[f], tie_break[f]))
        reps = list(reps)
        rng.shuffle(reps)
        for rep_key, fold in zip(reps, fold_order):
            fold_of[rep_key] = fold
            counts[fold] += 1
    return fold_of


def split_experiment_keys(experiments, split_seed, test_fold):
    keys = list(experiments)
    if not 0 <= test_fold < n_splits:
        raise ValueError(f"test_fold must be between 0 and {n_splits - 1}")

    test_fold_of = _assign_folds_no_combo_clustering(keys, n_splits, split_seed)
    val_fold_of = _assign_folds_no_combo_clustering(keys, n_splits, split_seed + 1)

    val_fold = (test_fold + 1) % n_splits
    train_keys, val_keys, test_keys = [], [], []
    for key in keys:
        if test_fold_of[key] == test_fold:
            test_keys.append(key)
        elif val_fold_of[key] == val_fold:
            val_keys.append(key)
        else:
            train_keys.append(key)
    return train_keys, val_keys, test_keys

### Experiments to Tensor

In [13]:
def experiments_to_tensor(keys, experiments, pipeline, window_size, stride, fit=False):
    ref_df = pd.concat([experiments[k] for k in keys], ignore_index=True)
    processed = pipeline.fit_transform(ref_df) if fit else pipeline.transform(ref_df)
 
    lengths = [len(experiments[k]) for k in keys]
 
    X_windows, y_windows = [], []
    start = 0
    for length in lengths:
        exp_slice = processed.iloc[start:start+length].reset_index(drop=True)
        start += length
 
        if len(exp_slice) < window_size:
            continue
 
        input_vals  = exp_slice[input_columns].values
        target_vals = exp_slice[target].values
 
        for i in range(0, len(exp_slice) - window_size + 1, stride):
            X_windows.append(input_vals[i:i+window_size])
            y_windows.append(target_vals[i+window_size-1])
 
    X = torch.tensor(np.array(X_windows), dtype=torch.float32)
    y = torch.tensor(np.array(y_windows), dtype=torch.float32).unsqueeze(1)
    return X, y

### Calculate Loss in Batches

In [14]:
def calculate_loss_in_batches(model, X, y, criterion, batch_size):
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=False)
    total_loss = 0.0
    total_samples = 0
    model.eval()
    with torch.no_grad():
        for X_batch, y_batch in loader:
            loss = criterion(model(X_batch), y_batch)
            total_loss += loss.item() * X_batch.size(0)
            total_samples += X_batch.size(0)
    return total_loss / total_samples

### Prepare Datasets and Train

In [15]:
def prepare_datasets(experiments, train_keys, val_keys, test_keys, window_size, stride):
    pipeline = Pipeline()
    X_train, y_train = experiments_to_tensor(train_keys, experiments, pipeline, window_size, stride, fit=True)
    X_val, y_val     = experiments_to_tensor(val_keys,   experiments, pipeline, window_size, stride, fit=False)
    X_test, y_test   = experiments_to_tensor(test_keys,  experiments, pipeline, window_size, stride, fit=False)
    print(f"Prepared tensors | train: {len(X_train):,} | val: {len(X_val):,} | test: {len(X_test):,} | window: {window_size} | stride: {stride}")
    return {
        'train_dataset': TensorDataset(X_train, y_train),
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test,
        'pipeline': pipeline,
    }

def train(model_class, model_name: str, prepared_data: dict, seed: int):
    set_seed(seed)
    print(f"\nTraining {model_name} ")
    model = model_class()
    criterion = nn.SmoothL1Loss(beta=1.0)
    val_criterion = nn.MSELoss()
    optimizer_class = getattr(torch.optim, model.optimizer)
    optimizer = optimizer_class(
        model.parameters(), lr=model.lr, weight_decay=model.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=scheduler_factor, patience=scheduler_patience
    )
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(prepared_data['train_dataset'], batch_size=model.batch_size, shuffle=True, generator=generator)

    best_val_loss = float('inf')
    best_state_dict = None
    best_epoch = -1
    epochs_without_improvement = 0

    for epoch in range(model.epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            if model.gradient_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), model.gradient_clip)
            optimizer.step()
            train_loss += loss.item()

        val_loss = calculate_loss_in_batches(
            model, prepared_data['X_val'], prepared_data['y_val'], val_criterion, model.batch_size
        )
        scheduler.step(val_loss)
        train_loss /= len(train_loader)
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"  Epoch {epoch + 1}/{model.epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {current_lr:.2e}")

        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_state_dict = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"  Early stopping at epoch {epoch + 1} (patience={patience})")
                break

    print(f"  Best epoch: {best_epoch} | Best Val Loss: {best_val_loss:.4f}")
    model.load_state_dict(best_state_dict)
    model.best_epoch = best_epoch
    model.best_val_loss = best_val_loss
    return model

### Inverse Transform Target

In [16]:
def inverse_transform_target(pipeline, values):
    n_cols = len(input_columns) + 1
    dummy = np.zeros((len(values), n_cols))
    dummy[:, -1] = values
    return pipeline.scaler.inverse_transform(dummy)[:, -1]

### Evaluate

In [17]:
def predict_scaled(model, X):
    loader = DataLoader(X, batch_size=model.batch_size, shuffle=False)
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X_batch).cpu() for X_batch in loader]).squeeze().numpy()


def evaluate_scaled_predictions(model_name, predictions, actuals, pipeline):
    predictions = inverse_transform_target(pipeline, predictions)
    actuals = inverse_transform_target(pipeline, actuals)
    mse = mean_squared_error(actuals, predictions)
    return {
        "Model": model_name,
        "MAE": mean_absolute_error(actuals, predictions),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(actuals, predictions),
    }


def evaluate(model, model_name, X_test, y_test, pipeline):
    performance = evaluate_scaled_predictions(
        model_name, predict_scaled(model, X_test), y_test.squeeze().numpy(), pipeline
    )
    performance["Best Epoch"] = model.best_epoch
    performance["Validation Loss"] = model.best_val_loss
    return performance

### Compute Experiment Predictions

In [18]:
def compute_experiment_predictions(model, key, experiments, pipeline):
    exp_df = experiments[key].copy()
    processed = pipeline.transform(exp_df)
 
    window_size = model.window_size
    stride = model.stride
    if len(processed) < window_size:
        return None
    input_vals  = processed[input_columns].values
    target_vals = processed[target].values
    depth_vals  = exp_df['depth'].values
    X_windows, y_windows, depth_windows = [], [], []
    for i in range(0, len(processed) - window_size + 1, stride):
        X_windows.append(input_vals[i:i+window_size])
        y_windows.append(target_vals[i+window_size-1])
        depth_windows.append(depth_vals[i+window_size-1])
    X = torch.tensor(np.array(X_windows), dtype=torch.float32)
    y = torch.tensor(np.array(y_windows), dtype=torch.float32).unsqueeze(1)
 
    model.eval()
    with torch.no_grad():
        preds = model(X).squeeze().numpy()
    actuals = y.squeeze().numpy()
 
    preds_orig   = inverse_transform_target(pipeline, preds)
    actuals_orig = inverse_transform_target(pipeline, actuals)
    depth_orig   = np.array(depth_windows)
 
    return depth_orig, actuals_orig, preds_orig

### Evaluate Test Experiments

In [19]:
def evaluate_test_experiments(model, model_name, test_keys, experiments, pipeline):
    rows = []
    for key in test_keys:
        result = compute_experiment_predictions(model, key, experiments, pipeline)
        if result is None:
            continue
        _, actuals, predictions = result
        mse = mean_squared_error(actuals, predictions)
        rows.append({
            "Density": key[0],
            "Cutting Speed": key[1],
            "Feed Rate": key[2],
            "Experiment ID": key[3],
            "Model": model_name,
            "MAE": mean_absolute_error(actuals, predictions),
            "RMSE": np.sqrt(mse),
        })
    return rows

### Plot Validation Overview

In [20]:
def plot_validation_overview(model, model_name, eval_keys, experiments, pipeline, output_root="validation_plots"):
    out_dir = os.path.join(output_root, model_name)
    os.makedirs(out_dir, exist_ok=True)
 
    exp_numbers, mean_actuals, mean_preds = [], [], []
 
    for idx, key in enumerate(eval_keys, start=1):
        result = compute_experiment_predictions(model, key, experiments, pipeline)
        if result is None:
            continue
        _, actuals_orig, preds_orig = result
        exp_numbers.append(idx)
        mean_actuals.append(np.mean(actuals_orig))
        mean_preds.append(np.mean(preds_orig))
 
    plt.figure(figsize=(10, 4))
    plt.plot(exp_numbers, mean_actuals, label="Actual",    color="steelblue", marker='o')
    plt.plot(exp_numbers, mean_preds,   label="Predicted", color="tomato", linestyle="--", marker='x')
    plt.title(f"{model_name} - Test Overview")
    plt.xlabel("Experiment Number")
    plt.ylabel("cutting_force (mean, unscaled)")
    plt.xticks(exp_numbers)
    plt.legend()
    plt.tight_layout()
 
    save_path = os.path.join(out_dir, f"{model_name}_test_overview.png")
    plt.savefig(save_path)
    plt.close()
    print(f"  Saved: {save_path}")

### Plot and Save per Experiment

In [21]:
def plot_and_save_per_experiment(model, model_name, eval_keys, experiments, pipeline, output_root="validation_plots"):
    out_dir = os.path.join(output_root, model_name)
    os.makedirs(out_dir, exist_ok=True)
 
    for key in eval_keys:
        result = compute_experiment_predictions(model, key, experiments, pipeline)
        if result is None:
            continue
        depth_orig, actuals_orig, preds_orig = result
 
        sort_idx       = np.argsort(depth_orig)
        depth_sorted   = depth_orig[sort_idx]
        actuals_sorted = actuals_orig[sort_idx]
        preds_sorted   = preds_orig[sort_idx]
 
        rmse_exp = np.sqrt(mean_squared_error(actuals_orig, preds_orig))
 
        exp_name  = "_".join(str(k) for k in key) if isinstance(key, tuple) else str(key)
        safe_name = exp_name.replace(" ", "").replace("/", "-")
 
        plt.figure(figsize=(10, 4))
        plt.plot(depth_sorted, actuals_sorted, label="Actual", color="steelblue", marker='o')
        plt.plot(depth_sorted, preds_sorted,   label="Predicted", color="tomato", linestyle="--", marker='x')
        plt.title(f"{model_name} RMSE: {rmse_exp:.4f} (Experiment {exp_name})")
        plt.xlabel("depth")
        plt.ylabel("cutting_force")
        plt.legend()
        plt.tight_layout()
 
        save_path = os.path.join(out_dir, f"{safe_name}.png")
        plt.savefig(save_path)
        plt.close()
        print(f"  Saved: {save_path}  |  RMSE: {rmse_exp:.4f}")

### LSTM

In [22]:
class LSTM(nn.Module):
    config       = model_configs["LSTM"]
    window_size  = config["window_size"]
    stride       = config["stride"]
    hidden_size  = config["hidden_size"]
    num_layers   = config["num_layers"]
    epochs       = epochs
    batch_size   = config["batch_size"]
    lr           = config["lr"]
    dropout      = config["dropout"]
    optimizer    = config["optimizer"]
    weight_decay = config["weight_decay"]
    gradient_clip = config["gradient_clip"]
 
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size, self.hidden_size, self.num_layers, batch_first=True, dropout=self.dropout)
        self.fc   = nn.Linear(self.hidden_size + input_size, output_size)
 
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        combined = torch.cat([h_n[-1], x[:, -1, :]], dim=1)
        return self.fc(combined)

### GRU

In [23]:
class GRU(nn.Module):
    config       = model_configs["GRU"]
    window_size  = config["window_size"]
    stride       = config["stride"]
    hidden_size  = config["hidden_size"]
    num_layers   = config["num_layers"]
    epochs       = epochs
    batch_size   = config["batch_size"]
    lr           = config["lr"]
    dropout      = config["dropout"]
    optimizer    = config["optimizer"]
    weight_decay = config["weight_decay"]
    gradient_clip = config["gradient_clip"]
 
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(input_size, self.hidden_size, self.num_layers, batch_first=True, dropout=self.dropout)
        self.fc  = nn.Linear(self.hidden_size + input_size, output_size)
 
    def forward(self, x):
        _, h_n = self.gru(x)
        combined = torch.cat([h_n[-1], x[:, -1, :]], dim=1)
        return self.fc(combined)

### RNN

In [24]:
class RNN(nn.Module):
    config       = model_configs["RNN"]
    window_size  = config["window_size"]
    stride       = config["stride"]
    hidden_size  = config["hidden_size"]
    num_layers   = config["num_layers"]
    epochs       = epochs
    batch_size   = config["batch_size"]
    lr           = config["lr"]
    dropout      = config["dropout"]
    optimizer    = config["optimizer"]
    weight_decay = config["weight_decay"]
    gradient_clip = config["gradient_clip"]
 
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size, self.hidden_size, self.num_layers, batch_first=True, dropout=self.dropout)
        self.fc  = nn.Linear(self.hidden_size + input_size, output_size)
 
    def forward(self, x):
        _, h_n = self.rnn(x)
        combined = torch.cat([h_n[-1], x[:, -1, :]], dim=1)
        return self.fc(combined)

### TCN

In [25]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.padding = padding
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding, dilation=dilation)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding, dilation=dilation)
        self.dropout = nn.Dropout(dropout)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def _causal_crop(self, x):
        return x[:, :, :-self.padding] if self.padding else x

    def forward(self, x):
        out = self._causal_crop(self.conv1(x))
        out = self.dropout(torch.relu(out))
        out = self._causal_crop(self.conv2(out))
        out = self.dropout(torch.relu(out))
        return torch.relu(out + self.residual(x))


class TCN(nn.Module):
    config = model_configs["TCN"]
    window_size = config["window_size"]
    stride = config["stride"]
    epochs = epochs
    batch_size = config["batch_size"]
    lr = config["lr"]
    optimizer = config["optimizer"]
    weight_decay = config["weight_decay"]
    gradient_clip = config["gradient_clip"]

    def __init__(self):
        super().__init__()
        channels = self.config["channels"]
        layers = []
        in_channels = input_size
        for index, out_channels in enumerate(channels):
            layers.append(TemporalBlock(in_channels, out_channels, self.config["kernel_size"], 2 ** index, self.config["dropout"]))
            in_channels = out_channels
        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(channels[-1] + input_size, output_size)

    def forward(self, x):
        features = self.network(x.transpose(1, 2))[:, :, -1]
        return self.fc(torch.cat([features, x[:, -1, :]], dim=1))

## Training

In [ ]:
configure_runtime()
df = pd.read_excel(data_path)
df = df.rename(columns=raw_column_map)

before = len(df)
df = df.drop_duplicates()
df = df.dropna()
after = len(df)
print(f"Dropped {before - after:,} rows (duplicates/empty values) | {before:,} -> {after:,}")

df = separate_experiments(df)
print_experiment_report(df)
experiments = get_experiments(df)
if run_single_model:
    train_keys, val_keys, test_keys = split_experiment_keys(experiments, split_seed, test_fold)
    config = model_configs["LSTM"]
    prepared_data = prepare_datasets(
        experiments, train_keys, val_keys, test_keys, config["window_size"], config["stride"]
    )

## LSTM

In [27]:
if run_single_model:
    lstm_model = train(LSTM, "LSTM", prepared_data, training_seed)
    lstm_performance = evaluate(lstm_model, "LSTM", prepared_data["X_test"], prepared_data["y_test"], prepared_data["pipeline"])

## GRU

In [28]:
if run_single_model:
    gru_model = train(GRU, "GRU", prepared_data, training_seed)
    gru_performance = evaluate(gru_model, "GRU", prepared_data["X_test"], prepared_data["y_test"], prepared_data["pipeline"])

## RNN

In [29]:
if run_single_model:
    rnn_model = train(RNN, "RNN", prepared_data, training_seed)
    rnn_performance = evaluate(rnn_model, "RNN", prepared_data["X_test"], prepared_data["y_test"], prepared_data["pipeline"])

## Evaluation

In [30]:
if run_single_model:
    performance_summary = pd.DataFrame([lstm_performance, gru_performance, rnn_performance]).set_index("Model")
    print(performance_summary.round(4))

### 5-Fold Evaluation

In [31]:
def run_fold(experiments, split_seed, training_seed, test_fold):
    model_specs = [(LSTM, "LSTM"), (GRU, "GRU"), (RNN, "RNN"), (TCN, "TCN")]
    individual_results = []
    experiment_results = []
    train_keys, val_keys, test_keys = split_experiment_keys(experiments, split_seed, test_fold)
    fold_output_root = os.path.join("validation_plots_raw", f"Fold {test_fold + 1}")

    print(
        f"5-fold stratified split | test fold: {test_fold + 1}/{n_splits} | "
        f"train: {len(train_keys)} experiments | "
        f"validation: {len(val_keys)} | test: {len(test_keys)}"
    )

    for model_class, model_name in model_specs:
        config = model_configs[model_name]
        prepared_data = prepare_datasets(
            experiments,
            train_keys,
            val_keys,
            test_keys,
            config["window_size"],
            config["stride"],
        )
        print(f"\n{'=' * 16} {model_name} {'=' * 16}")
        model = train(model_class, model_name, prepared_data, training_seed)
        test_prediction = predict_scaled(model, prepared_data["X_test"])
        validation_prediction = predict_scaled(model, prepared_data["X_val"])
        performance = evaluate_scaled_predictions(
            model_name,
            test_prediction,
            prepared_data["y_test"].squeeze().numpy(),
            prepared_data["pipeline"],
        )
        performance["Best Epoch"] = model.best_epoch
        performance["Validation Loss"] = model.best_val_loss
        validation_performance = evaluate_scaled_predictions(
            model_name,
            validation_prediction,
            prepared_data["y_val"].squeeze().numpy(),
            prepared_data["pipeline"],
        )
        performance["Validation RMSE"] = validation_performance["RMSE"]
        performance["Training Seed"] = training_seed
        individual_results.append(performance)
        experiment_results.extend(
            evaluate_test_experiments(
                model, model_name, test_keys, experiments, prepared_data["pipeline"]
            )
        )

        plot_validation_overview(
            model, model_name, test_keys, experiments, prepared_data["pipeline"], output_root=fold_output_root
        )
        plot_and_save_per_experiment(
            model, model_name, test_keys, experiments, prepared_data["pipeline"], output_root=fold_output_root
        )

    return pd.DataFrame(individual_results), pd.DataFrame(experiment_results)


def run_cross_validation(experiments, split_seed, training_seed):
    individual_folds = []
    experiment_folds = []

    for test_fold in range(n_splits):
        individual_results, experiment_results = run_fold(
            experiments, split_seed, training_seed, test_fold
        )
        individual_results["Test Fold"] = test_fold + 1
        experiment_results["Test Fold"] = test_fold + 1
        individual_folds.append(individual_results)
        experiment_folds.append(experiment_results)

    return (
        pd.concat(individual_folds, ignore_index=True),
        pd.concat(experiment_folds, ignore_index=True),
    )


def summarize_cross_validation(results):
    return results.groupby("Model", sort=False).agg(
        **{
            "MAE Mean": ("MAE", "mean"),
            "MSE Mean": ("MSE", "mean"),
            "RMSE Mean": ("RMSE", "mean"),
            "RMSE Std": ("RMSE", "std"),
            "R² Mean": ("R2", "mean"),
            "Validation RMSE Mean": ("Validation RMSE", "mean"),
        }
    ).sort_values("RMSE Mean")


def summarize_worst_experiments(results):
    columns = ["Density", "Cutting Speed", "Feed Rate", "Experiment ID"]
    return results.groupby(columns, as_index=False).agg(
        **{
            "Mean MAE": ("MAE", "mean"),
            "Mean RMSE": ("RMSE", "mean"),
            "Maximum RMSE": ("RMSE", "max"),
            "Evaluated Models": ("Model", "nunique"),
        }
    ).sort_values("Mean RMSE", ascending=False).head(10)


cross_validation_results, experiment_results = run_cross_validation(
    experiments, split_seed, training_seed
)
cross_validation_summary = summarize_cross_validation(cross_validation_results)
worst_experiments = summarize_worst_experiments(experiment_results)
display(worst_experiments.round(4))
display(cross_validation_summary.round(4))

5-fold stratified split | test fold: 1/5 | train: 127 experiments | validation: 32 | test: 39
Prepared tensors | train: 33,491 | val: 9,094 | test: 10,281 | window: 100 | stride: 5

================ LSTM ================

Training LSTM 
  Epoch 1/30 | Train Loss: 0.0150 | Val Loss: 0.0049 | LR: 1.00e-03
  Epoch 2/30 | Train Loss: 0.0029 | Val Loss: 0.0033 | LR: 1.00e-03
  Epoch 3/30 | Train Loss: 0.0019 | Val Loss: 0.0053 | LR: 1.00e-03
  Epoch 4/30 | Train Loss: 0.0018 | Val Loss: 0.0283 | LR: 1.00e-03
  Epoch 5/30 | Train Loss: 0.0019 | Val Loss: 0.0024 | LR: 1.00e-03
  Epoch 6/30 | Train Loss: 0.0016 | Val Loss: 0.0035 | LR: 1.00e-03
  Epoch 7/30 | Train Loss: 0.0014 | Val Loss: 0.0035 | LR: 1.00e-03
  Epoch 8/30 | Train Loss: 0.0012 | Val Loss: 0.0028 | LR: 5.00e-04
  Epoch 9/30 | Train Loss: 0.0010 | Val Loss: 0.0023 | LR: 5.00e-04
  Epoch 10/30 | Train Loss: 0.0009 | Val Loss: 0.0024 | LR: 5.00e-04
  Epoch 11/30 | Train Loss: 0.0010 | Val Loss: 0.0024 | LR: 5.00e-04
  Epoch 12/30

,Density,Cutting Speed,Feed Rate,Experiment ID,Mean MAE,Mean RMSE,Maximum RMSE,Evaluated Models
108,20,10,10,1,59.2111,66.6020,71.9552,4
173,25,16,20,2,34.6399,40.8952,50.2115,4
163,25,10,10,2,32.0938,36.8805,40.9659,4
162,25,10,10,1,28.8501,36.5721,46.1077,4
112,20,10,15,2,25.8550,31.9007,43.6765,4
168,25,16,10,1,23.4843,27.3268,35.7540,4
139,20,40,15,2,20.7226,27.0409,50.6514,4
110,20,10,10,3,20.5813,25.9405,46.0440,4
119,20,16,10,3,21.2420,25.4056,33.2552,4
115,20,10,20,2,18.2881,22.9469,28.7289,4


,MAE Mean,MSE Mean,RMSE Mean,RMSE Std,R² Mean,Validation RMSE Mean
Model,,,,,,
LSTM,8.1384,163.2174,12.5802,2.4892,0.9965,11.5938
GRU,8.6003,179.1207,13.2875,1.7897,0.9960,12.3686
RNN,9.0962,188.2706,13.5834,2.1683,0.9959,12.6325
TCN,9.1989,211.6952,14.3781,2.4917,0.9955,13.7463
